<a href="https://colab.research.google.com/github/hakim733/Neural_Network/blob/main/simle_nn_lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB1 : Abdelhakim Mraihi
1.   To Run This Code (Outside Notebooks):
Install Dependencies: Open a terminal and run:
"pip install torch pandas scikit-learn matplotlib" and
Run the script:"python simle_nn.py" If you’ve saved the code as simle_nn.py, run:

2. Dataset Description:
The dataset used is the Iris dataset, loaded from a public URL:
https://gist.githubusercontent.com/netj/8836201/raw/.../iris.csv
It contains 150 samples of iris flowers with:

  4 numerical features (sepal length, sepal width, petal length, petal width)

  A categorical target with 3 classes (Setosa, Versicolor, Virginica)

  This is a multi-class classification problem.

3. Libraries Used:
   Here are all the libraries used with install commands:
- pip install torch          # PyTorch for building and training the neural network
- pip install pandas         # For data handling
- pip install scikit-learn   # For preprocessing and train-test split
- pip install matplotlib     # Optional: for plotting if desired
No unusual or custom libraries beyond that.






In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math

In [ ]:
# Create a Model Class that inherits nn.Module
class Model(nn.Module):
  # Input layer (4 features of the flower) -->
  # Hidden Layer1 (number of neurons) -->
  # H2 (n) -->H3(n) -->H4(n)
  # output (3 classes of iris flowers)
  def __init__(self, in_features=4, h1=8, h2=9, h3=5, out_features=3):
    super().__init__() # instantiate our nn.Module
    self.fc1 = nn.Linear(in_features, h1) #Starting from th first layer and moving to the second
    self.fc2 = nn.Linear(h1, h2)
    self.fc3 = nn.Linear(h2, h3)
    self.out = nn.Linear(h3, out_features)

  def forward(self, x):   # for forward propagation i chosed Relu
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = self.out(x)

    return x


In [ ]:
# Pick a manual seed for randomization, i choosed 41 where the NN will start
torch.manual_seed(41)
# Create an instance of model
model = Model()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
url = 'https://gist.githubusercontent.com/netj/8836201/raw/6f9306ad21398ea43cba4f7d537619d0e07d5ae3/iris.csv'
my_df = pd.read_csv(url)


In [ ]:
my_df.tail()

,sepal.length,sepal.width,petal.length,petal.width,variety
145,6.7,3.0,5.2,2.3,Virginica
146,6.3,2.5,5.0,1.9,Virginica
147,6.5,3.0,5.2,2.0,Virginica
148,6.2,3.4,5.4,2.3,Virginica
149,5.9,3.0,5.1,1.8,Virginica


In [ ]:
# Change last column from strings to integers
my_df['variety'] = my_df['variety'].astype('category').cat.codes

In [ ]:
# Train Test Split!  Set X, y
X = my_df.drop('variety', axis=1)
y = my_df['variety']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=41)

In [ ]:
# Convert X features to float tensors
X_train = torch.FloatTensor(X_train.values)
X_test = torch.FloatTensor(X_test.values)
my_df = torch.FloatTensor(my_df.values)

In [ ]:
# Convert y labels to tensors long
#y_train = torch.LongTensor(y_train.to_numpy())
#y_test = torch.LongTensor(y_test.to_numpy())

y_train = torch.LongTensor(y_train.to_numpy())  # or .values
y_test = torch.LongTensor(y_test.to_numpy())


In [ ]:
# Create dataset and dataloader
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train, y_train)

dataloader = DataLoader(dataset = train_dataset, batch_size=4, shuffle=True, num_workers=2)

dataiter = iter(dataloader)
batch = next(dataiter)
print(batch)

[tensor([[4.8000, 3.4000, 1.9000, 0.2000],
        [6.3000, 3.4000, 5.6000, 2.4000],
        [5.1000, 2.5000, 3.0000, 1.1000],
        [5.7000, 2.8000, 4.5000, 1.3000]]), tensor([0, 2, 1, 1])]


In [ ]:
#train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
#test_loader = DataLoader(test_dataset, batch_size=16)


In [ ]:
# Set the criterion of model to measure the error, how far off the predictions are from the data
criterion = nn.CrossEntropyLoss()
# Choose Adam Optimizer, lr = learning rate (if error doesn't go down after a bunch of iterations (epochs), lower our learning rate)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
model.parameters


<bound method Module.parameters of Model(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=9, bias=True)
  (fc3): Linear(in_features=9, out_features=5, bias=True)
  (out): Linear(in_features=5, out_features=3, bias=True)
)>

In [ ]:
# Train our model!
# Epochs? (one run thru all the training data in our network)
epochs = 100
losses = []
for i in range(epochs):
  # Go forward and get a prediction
  y_pred = model.forward(X_train) # Get predicted results

  # Measure the loss/error, gonna be high at first
  loss = criterion(y_pred, y_train) # predicted values vs the y_train

  # Keep Track of our losses
  losses.append(loss.detach().numpy())

  # print every 10 epoch
  if i % 10 == 0:
    print(f'Epoch: {i} and loss: {loss}')

  # Do some back propagation: take the error rate of forward propagation and feed it back
  # thru the network to fine tune the weights
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()


Epoch: 0 and loss: 1.1593784093856812
Epoch: 10 and loss: 1.0903542041778564
Epoch: 20 and loss: 1.0583134889602661
Epoch: 30 and loss: 0.9496428370475769
Epoch: 40 and loss: 0.6863871812820435
Epoch: 50 and loss: 0.4488547742366791
Epoch: 60 and loss: 0.27205365896224976
Epoch: 70 and loss: 0.12178926169872284
Epoch: 80 and loss: 0.057799242436885834
Epoch: 90 and loss: 0.03869441896677017


In [ ]:
# Evaluate Model on Test Data Set (validate model on test set)
with torch.no_grad():  # Basically turn off back propogation
  y_eval = model.forward(X_test) # X_test are features from our test set, y_eval will be predictions
  loss = criterion(y_eval, y_test) # Find the loss or error

In [ ]:
loss

tensor(0.1262)

In [ ]:
correct = 0
with torch.no_grad():
  for i, data in enumerate(X_test):
    y_val = model.forward(data)

    if y_test[i] == 0:
      x = "Setosa"
    elif y_test[i] == 1:
      x = 'Versicolor'
    else:
      x = 'Virginica'


    # Will tell us what type of flower class our network thinks it is
    print(f'{i+1}.)  {str(y_val)} \t {y_test[i]} \t {y_val.argmax().item()}')

    # Correct or not
    if y_val.argmax().item() == y_test[i]:
      correct +=1

print(f'We got {correct} correct!')

1.)  tensor([-11.5322,   7.6509,  10.8490]) 	 2 	 2
2.)  tensor([-16.8627,   8.9481,  18.1188]) 	 2 	 2
3.)  tensor([-18.4435,  10.1912,  19.4110]) 	 2 	 2
4.)  tensor([-5.3411,  6.9027,  1.6414]) 	 1 	 1
5.)  tensor([-15.0067,   8.9891,  15.0912]) 	 2 	 2
6.)  tensor([-2.3246,  6.3972, -2.7044]) 	 1 	 1
7.)  tensor([-11.3070,   7.9685,  10.1658]) 	 2 	 2
8.)  tensor([-5.1876,  6.9088,  1.3876]) 	 1 	 1
9.)  tensor([-13.0754,   8.4640,  12.5122]) 	 2 	 2
10.)  tensor([-17.9465,   9.5009,  19.3063]) 	 2 	 2
11.)  tensor([-10.3891,   7.7352,   8.9241]) 	 2 	 2
12.)  tensor([  9.2140,   3.6189, -18.7334]) 	 0 	 0
13.)  tensor([  8.3127,   3.2089, -16.8337]) 	 0 	 0
14.)  tensor([ 0.2088,  5.0874, -5.4626]) 	 1 	 1
15.)  tensor([  9.2589,   3.6393, -18.8280]) 	 0 	 0
16.)  tensor([-9.1893,  7.6120,  7.1194]) 	 2 	 1
17.)  tensor([  8.7912,   3.4265, -17.8421]) 	 0 	 0
18.)  tensor([-10.9128,   7.6573,   9.8463]) 	 1 	 2
19.)  tensor([  9.0244,   3.5326, -18.3337]) 	 0 	 0
20.)  tensor([  8

In [ ]:
num_epochs = 2
total_samples = len(my_df)
n_iterations = math.ceil(total_samples/4)
print(total_samples, n_iterations)

150 38


In [ ]:
for epoch in range(num_epochs):
  for i, (features, labels) in enumerate(dataloader):
    #forward, backward pass and update the wights
    if(i+1)%5 == 0:
     print(f'epoch {epoch+1}/{num_epochs}, step {i+1}/{n_iterations}, inputs {X_train.shape}')



epoch 1/2, step 5/38, inputs torch.Size([120, 4])
epoch 1/2, step 10/38, inputs torch.Size([120, 4])
epoch 1/2, step 15/38, inputs torch.Size([120, 4])
epoch 1/2, step 20/38, inputs torch.Size([120, 4])
epoch 1/2, step 25/38, inputs torch.Size([120, 4])
epoch 1/2, step 30/38, inputs torch.Size([120, 4])
epoch 2/2, step 5/38, inputs torch.Size([120, 4])
epoch 2/2, step 10/38, inputs torch.Size([120, 4])
epoch 2/2, step 15/38, inputs torch.Size([120, 4])
epoch 2/2, step 20/38, inputs torch.Size([120, 4])
epoch 2/2, step 25/38, inputs torch.Size([120, 4])
epoch 2/2, step 30/38, inputs torch.Size([120, 4])
